In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)
feature_dataset = pd.read_csv("../data/processed/feature_dataset.csv")
feature_dataset.head()
feature_dataset.shape
X = feature_dataset.drop("SepsisLabel", axis=1)
y = feature_dataset["SepsisLabel"]
X_train, X_test, y_train , y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    stratify = y,
    random_state=42
)
#CROSS VALIDATION
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

def objective(trial):

    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves", 20, 150
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 15
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples", 5, 100
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 10.0, log=True
        ),

        "random_state": 42
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(
        model,
        X_train,     # this denotes the vitals
        y_train,     # this denotes the labels
        cv = cv,     # denotes stratifiedkfold used for 5 folds
        scoring = "f1",
        n_jobs = -1    # this is used so tht the all 5 models work simultaneously
    )
    return scores.mean()

study = optuna.create_study(direction = "maximize")
study.optimize(
    objective,
    n_trials = 50
)
print("Best F1 Score:", study.best_value)
print("Best Parameters:")
print(study.best_params)

C:\Users\prajin\PycharmProjects\Explainable-Sepsis-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-13 10:44:59,684] A new study created in memory with name: no-name-4f605889-0824-4f1e-b611-9bbf9f2d131d
[I 2026-08-13 10:45:17,956] Trial 0 finished with value: 0.7745623743482605 and parameters: {'learning_rate': 0.03880077887889892, 'n_estimators': 699, 'num_leaves': 119, 'max_depth': 9, 'min_child_samples': 43, 'subsample': 0.9965455732653642, 'colsample_bytree': 0.98495474112929, 'reg_alpha': 0.2614347443891571, 'reg_lambda': 1.2690962919193057e-06}. Best is trial 0 with value: 0.7745623743482605.
[I 2026-08-13 10:45:25,257] Trial 1 finished with value: 0.7668591138772729 and parameters: {'learning_rate': 0.20314801603697924, 'n_estimators': 604, 'num_leaves': 146, 'max_depth': 3, 'min_child_sa

Best F1 Score: 0.781601105984395
Best Parameters:
{'learning_rate': 0.06406571878171621, 'n_estimators': 790, 'num_leaves': 52, 'max_depth': 6, 'min_child_samples': 52, 'subsample': 0.7303346534096313, 'colsample_bytree': 0.931238032924673, 'reg_alpha': 6.8282389288986085, 'reg_lambda': 5.0247093154677035e-05}


In [2]:
best_model = lgb.LGBMClassifier(
    **study.best_params,
    objective="binary",
    random_state=42
)

best_model.fit(X_train, y_train)

print("Optimized model trained successfully!")


[LightGBM] [Info] Number of positive: 1432, number of negative: 14836
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010787 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 31062
[LightGBM] [Info] Number of data points in the train set: 16268, number of used features: 192
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.088026 -> initscore=-2.337985
[LightGBM] [Info] Start training from score -2.337985
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [3]:
y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)[:, 1]

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("AUROC    :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.9614060963618486
Precision: 0.900398406374502
Recall   : 0.6312849162011173
F1 Score : 0.7422003284072249
AUROC    : 0.9469062928217561

Confusion Matrix
[[3685   25]
 [ 132  226]]
